<div style="border-top:4px solid #0f766e;padding:28px 0 18px"><div style="color:#0f766e;font-weight:700;letter-spacing:.8px">模块 13：存储与生命周期管理</div><div style="color:#17212b;font-size:30px;font-weight:750">模块 13：存储与生命周期管理</div><p style="color:#475569;line-height:1.7">通过分区、Tablet 和物理布局证据理解存储生命周期。请按顺序运行；结果会以表格展示，写入只作用于本模块的 `_l3` 对象。</p></div>

## 边界

本实验只创建和维护 `ops_orders_l3`，不会截断 `orders_imported` 或其他共享表。

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "dw_course").is_dir())
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course import WarehouseLab

lab = WarehouseLab()


In [ ]:
lab.execute("DROP TABLE IF EXISTS ops_orders_l3")
lab.execute("""
CREATE TABLE ops_orders_l3 (
    order_date DATE NOT NULL,
    order_id BIGINT NOT NULL,
    status VARCHAR(20) NOT NULL,
    amount DECIMAL(18,2) NOT NULL
)
DUPLICATE KEY(order_date, order_id)
PARTITION BY RANGE(order_date) (
    PARTITION p202501 VALUES LESS THAN ("2025-02-01"),
    PARTITION p202502 VALUES LESS THAN ("2025-03-01"),
    PARTITION p202503 VALUES LESS THAN ("2025-04-01")
)
DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES ("replication_num"="1")
""")
lab.insert("ops_orders_l3", ["order_date", "order_id", "status", "amount"], [("2025-01-15", 130001, "PAID", 100), ("2025-02-15", 130002, "PAID", 200), ("2025-03-15", 130003, "SHIPPED", 300)])
lab.sql("SELECT * FROM ops_orders_l3 ORDER BY order_date, order_id", title="隔离的生命周期表")

In [ ]:
lab.sql("SHOW PARTITIONS FROM ops_orders_l3 ORDER BY PartitionName", title="分区与行数证据")
lab.sql("SHOW TABLETS FROM ops_orders_l3", title="Tablet 布局证据")
lab.sql("SHOW CREATE TABLE ops_orders_l3", title="保留策略与分布契约")

In [ ]:
lab.execute("TRUNCATE TABLE ops_orders_l3 PARTITION (p202501)")
lab.sql("SELECT * FROM ops_orders_l3 ORDER BY order_date, order_id", title="清理一月数据后的剩余数据")
lab.sql("SHOW PARTITIONS FROM ops_orders_l3 ORDER BY PartitionName", title="维护后的元数据")

## 要点

分区级维护可以限制逻辑影响范围；逻辑可见性与物理空间回收需要分别检查。